# 7000LBSBAI - Programming for Business Analytics
## Week 9 - Task: FitLife Over Time, by Segment, and a First Model
### 📋 ANSWER VERSION: for checking code and independent study

---

**Module:** Programming for Business Analytics  
**Week:** 9  
**Time:** About 75 minutes, plus optional extensions  
**Topics covered:** Creating an outcome from a date · Time series · Pivot tables of rates · Gemini as an assistant · Linear regression

---

### How to use this notebook

| Symbol | Meaning |
|--------|---------|
| ✍️ Worked example | Code is already written -> run it and study the output |
| ➡️ Your turn | Write or complete the code in this cell |
| 💬 What does it mean? | Write a short interpretation of your results |
| 🤖 AI activity | A guided activity using Gemini in Colab |
| 🔴 Extension | For students who finish early (optional) |
| 📝 Hint/Tip | Some friendly help and advice |

To run a cell: click on it and press **Shift + Enter**.

---

## ⚠️ BEFORE YOU CONTINUE

### Save a copy of this notebook

Make sure your notebook is saved to your Google Drive:
- **File → Save a copy in Drive**
- Name it: `Week9_Task_FitLife_Time_Segments_Regression_YourName.ipynb`

---

### ✨ Unhide generative AI features (keep inline completions turned off)

Go to **Tools** -> **Settings** -> **AI Assistance**

> Uncheck `☐ Show AI-powered inline completions`  
>
> Uncheck `☐ Hide generative AI features`

---
## The scenario

You are continuing your analysis of **FitLife Gyms** from Week 8. The membership manager now asks:

> ### *"How have joins and cancellations changed over time, which combinations of members are most at risk, and what drives how much members spend?"*

> 📝 Hints say which **code-along block** or **week** covers each skill. Some questions are open: decide how to investigate, then explain your choice.

---
## Set up: run this cell

This cell loads FitLife and cleans it with the Week 8 function. It also converts the dates and works out `tenure_months` (how long each member has belonged), as you did in Week 8.

In [ ]:
# ✍️ Set up: just run this cell
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.formula.api as smf

sns.set_theme(style='whitegrid')

def clean_members(data):
    """Clean the FitLife member data (the Week 8 function)."""
    data = data.copy()
    data = data.drop_duplicates()
    data['membership_type'] = data['membership_type'].str.strip().str.title()
    data['monthly_fee'] = pd.to_numeric(data['monthly_fee'].str.replace('£', ''))
    data = data[(data['age'] >= 18) & (data['age'] <= 100)]
    data['visits_per_month'] = data['visits_per_month'].fillna(data['visits_per_month'].median())
    return data

df = clean_members(pd.read_csv("https://raw.githubusercontent.com/7000lbsbai/7000LBSBAI/main/datasets/fitlife_members.csv"))

df['join_date'] = pd.to_datetime(df['join_date'])
df['cancel_date'] = pd.to_datetime(df['cancel_date'])
end_date = df['cancel_date'].fillna(pd.Timestamp('2025-01-31'))
df['tenure_months'] = ((end_date - df['join_date']).dt.days / 30.44).round(1)

print(f"Data ready: {df.shape[0]} members")

---
# Section A: Create the outcome from a date
*about 10 minutes*

In real data, the outcome you want to study is not always a ready-made column. Here, a member has cancelled if they have a `cancel_date`. FitLife also has a `cancelled` column, so you can **check** your version against it.

> 📝 **Hint:** Week 8. `.isna()` is True where a value is missing.

In [ ]:
# ➡️ A.1 Create a column 'cancelled_check': 'Yes' if the member has a cancel_date, 'No' if not. Then check it matches the cancelled column for every member
df['cancelled_check'] = np.where(df['cancel_date'].isna(), 'No', 'Yes')

print(f"Matches cancelled for every member: {(df['cancelled_check'] == df['cancelled']).all()}")
pd.crosstab(df['cancelled_check'], df['cancelled'])

# ✅ True: 2,300 No and 697 Yes, with no disagreements

In [ ]:
# ➡️ A.2 Create a 1/0 column cancelled_flag from cancel_date, and print the cancellation rate (%)
df['cancelled_flag'] = np.where(df['cancel_date'].notna(), 1, 0)
print(f"Cancellation rate: {df['cancelled_flag'].mean() * 100:.1f}%")

# ✅ 23.3%, the same as Week 8

### ✏️ A.3 Why check?

- Why is it important to **check** an outcome you have created yourself?
- What could go wrong if you **filled** the empty cancel dates before creating it?

**MODEL ANSWER:**

A created outcome is the foundation of every later result (every rate, chart and model), so a mistake here would silently spoil the whole analysis. Checking it against another source, or at least checking that the counts make sense, catches this early. If the empty cancel dates were **filled** first (for example with the date the data was taken), **every** member would appear to have cancelled: the rate would be 100% and every pattern would disappear. Empty cancel dates are not errors: they mean the member has not cancelled.

---
# Section B: Patterns over time
*about 20 minutes*

In [ ]:
# ➡️ B.1 Count the joins in each month and draw a line chart, with a title and axis labels
joins = df.set_index('join_date').resample('ME').size()
print(joins.describe().round(1))

plt.figure(figsize=(12, 5))
plt.plot(joins.index, joins.values, marker='o')
plt.title('New FitLife members each month')
plt.xlabel('Month')
plt.ylabel('Members joining')
plt.show()

# ✅ Steady: about 83 a month (65 to 103), with no clear trend or seasonal peak

> 📝 **Hint:** code-along Block 1. `.size()` counts the rows in each month (there is no column to add up).

In [ ]:
# ➡️ B.2 Count the cancellations in each month (members who cancelled only), and draw a line chart with a 3-month rolling average
cancels = df.dropna(subset=['cancel_date']).set_index('cancel_date').resample('ME').size()
print(cancels.tail(6))

plt.figure(figsize=(12, 5))
plt.plot(cancels.index, cancels.values, marker='o', alpha=0.6, label='Cancellations')
plt.plot(cancels.index, cancels.rolling(3).mean().values, linewidth=3, label='3-month rolling average')
plt.title('FitLife cancellations each month')
plt.xlabel('Month')
plt.ylabel('Cancellations')
plt.legend()
plt.show()

# ✅ Rising slowly as membership grows (about 20 a month in 2024), then a huge spike: 116 in January 2025

### ➡️ B.3 Is that spike real?

Something unusual happens in the **last month**. Before you report it, **investigate**: which **dates** have the most cancellations?

> 📝 **Hint:** `value_counts()` on `cancel_date`.

In [ ]:
# ➡️ B.3 Investigate the spike
print(df['cancel_date'].value_counts().head())

# ✅ 95 cancellations are dated 30 January 2025, the day before the data was taken.
#    No other day has more than 4

### ✏️ B.4 What do you think happened?

- Is the January 2025 spike a **real** change in member behaviour? What is the evidence?
- What would you do about it before reporting monthly cancellations?

**MODEL ANSWER:**

Almost certainly **not real**. 95 of the 116 January cancellations are on a **single day**, 30 January 2025, the day before the data was taken, while no other day in three years has more than 4. A genuine change in behaviour (for example after a price rise) would build up over days or weeks. This looks like a **data recording issue**: for example, cancellations being processed in a batch and given the processing date. Before reporting, I would **ask the data owner** what happened, **flag** the spike in the report rather than present it as a finding, and leave January 2025 out of any trend (for example, stop the monthly chart at December 2024). Always investigate a surprising pattern before reporting it.

In [ ]:
# ➡️ B.5 When do cancellations happen? For members who cancelled, count the cancellations in each month of membership (0, 1, 2…) and draw a bar chart
cancelled_members = df[df['cancelled_flag'] == 1]
by_month = cancelled_members['tenure_months'].astype(int).value_counts().sort_index()
print(by_month.head(10))

plt.figure(figsize=(12, 5))
plt.bar(by_month.index, by_month.values)
plt.title('Cancellations by month of membership')
plt.xlabel('Months since joining')
plt.ylabel('Cancellations')
plt.show()

# ✅ Most cancellations happen in months 1 to 4 (around 90 each), then fall steadily

> 📝 **Hint:** `.astype(int)` turns 4.6 months into 4 (four complete months). **For Assessment 2:** NovaTel has `Tenure_Months`, so you can do the same kind of analysis there.

---
# Section C: Two factors at once
*about 15 minutes*

In Week 8, membership type and visits were both linked to cancellation. Do they **combine**?

1. Create `visits_band`: four equal-sized groups of `visits_per_month` (Week 8).
2. Create a pivot table of the **cancellation rate (%)** with `membership_type` as the rows and `visits_band` as the columns, and a second pivot table of the **counts**.
3. Draw a heatmap of the rates.

> 📝 **Hint:** code-along Block 2.

In [ ]:
# ➡️ C.1 A pivot table of cancellation rates by membership type and visits band, the counts, and a heatmap
df['visits_band'] = pd.qcut(df['visits_per_month'], 4, labels=['Lowest 25%', 'Second', 'Third', 'Highest 25%'])

rates = df.pivot_table(values='cancelled_flag', index='membership_type', columns='visits_band', aggfunc='mean', observed=True) * 100
print(rates.round(1))
print()
print(df.pivot_table(values='cancelled_flag', index='membership_type', columns='visits_band', aggfunc='count', observed=True))
print()

plt.figure(figsize=(9, 5))
sns.heatmap(rates, annot=True, fmt='.1f', cmap='Reds', vmin=0, vmax=60)
plt.title('Cancellation rate (%), by membership type and visits')
plt.xlabel('Visits per month')
plt.ylabel('Membership type')
plt.show()

# ✅ From 56.2% (Monthly Flex, lowest visits) to 1.4% (Annual, highest visits). Every cell has at least 87 members

### 💬 What does it mean?

- Which combination of members is most at risk, and which is least?
- Do membership type and visits **both** matter, even when you look at them together?
- Which group would you target first with a retention offer, and why?

**✏️ MODEL ANSWER:**

The most at-risk members are **Monthly Flex members who rarely visit**: **56.2%** have cancelled. The least at risk are **Annual members who visit most**: just **1.4%**. **Both factors matter together**: within every membership type, cancellation falls steadily as visits rise, and within every visits band, Monthly Flex is highest and Annual lowest. Every cell has at least 87 members, so the rates are reliable. I would target **low-visiting Monthly Flex members** first: it is the largest high-risk group (464 members), and encouraging them to visit, or offering a discounted move to an Annual membership, could reduce cancellations.

---
# Section D: 🤖 Gemini as an assistant
*about 15 minutes*

Last week Gemini was your **tutor**: it explained code. This week it is your **assistant**: it can **write** code for you, but **you** are responsible for checking it.

### This week's rules

| ✅ You may… | ⚠️ You must… |
|---|---|
| Ask Gemini to write code for a task you describe | Read and explain **every line** before you run it |
| Use the code if it is correct | **Check** the result against something you already know |
| | Never use code you cannot explain |
| | Record what happened in your AI log |

### The activity

1. Open the **Gemini** chat panel (✨). Make sure inline completions are still **off** (Week 8).

2. Paste this prompt:

> *"I have a pandas DataFrame called df. The column cancel_date has already been converted with pd.to_datetime, and it is empty for members who have not cancelled. Write Python code that counts the cancellations in each month and draws a line chart of the monthly counts with a 3-month rolling average, a title and axis labels. Use pandas and matplotlib only."*

3. Select ✔️ Accept. **Before running it**, check the `#` comment above each line saying what it does, and check you understand each section of code. If necessary, rewrite the `#` comment in your own words.

4. Run it, then **check** it in the next cell: does it give the same monthly counts as your answer to B.2?

In [ ]:
# 🤖 D.1 Gemini's code, with your own comment above each line
# (Gemini's code will vary. A typical correct version:)

# Keep only the members who cancelled, and use cancel_date as the index
cancelled_only = df.dropna(subset=['cancel_date']).set_index('cancel_date')
# Count the cancellations in each month
monthly_cancellations = cancelled_only.resample('ME').size()
# A 3-month rolling average of the monthly counts
rolling_3m = monthly_cancellations.rolling(window=3).mean()

# Draw both lines on one chart
plt.figure(figsize=(12, 6))
plt.plot(monthly_cancellations.index, monthly_cancellations.values, label='Monthly cancellations')
plt.plot(rolling_3m.index, rolling_3m.values, label='3-month rolling average')
plt.title('Monthly cancellations with a 3-month rolling average')
plt.xlabel('Month')
plt.ylabel('Number of cancellations')
plt.legend()
plt.show()

In [ ]:
# 🤖 D.2 Check: compare Gemini's monthly counts with yours from B.2
print(f"Same counts in every month: {(monthly_cancellations == cancels).all()}")
print(f"Total cancellations: Gemini {monthly_cancellations.sum()}, mine {cancels.sum()}, members who cancelled {df['cancelled_flag'].sum()}")

# ✅ True, and all three totals are 697.
# Things to watch for in Gemini's code: did it drop the members with no cancel_date?
# Did it use 'M' instead of 'ME' (which gives a warning in newer pandas)?

> 📝 **Checking ideas:** compare the totals (they should equal the number of members who cancelled), compare one or two months with B.2, and check that January 2025 shows the same spike.

### 📓 D.3 Add this to your AI log

Record what you asked, what Gemini produced, **how you checked it**, and anything it got wrong or did differently from you. If you started an AI log in Week 8, add it to this; if not, start an AI log now.

---
# Section E: Adding a regression model
*about 15 minutes*

What drives how much members **spend** each month? Start with one predictor: visits.

> 📝 **Hint:** code-along Block 4.

In [ ]:
# ➡️ E.1 Fit a model of monthly_spend explained by visits_per_month. Print the coefficient, its p-value and R², and predict the spend of members who visit 4 and 12 times a month
spend_model = smf.ols('monthly_spend ~ visits_per_month', data=df).fit()

print(f"Coefficient: £{spend_model.params['visits_per_month']:.2f} per visit")
print(f"p-value:     {spend_model.pvalues['visits_per_month']:.4f}")
print(f"R-squared:   {spend_model.rsquared:.2f}")

new = pd.DataFrame({'visits_per_month': [4, 12]})
print("\n", spend_model.predict(new).round(2))

# ✅ £2.67 per visit; p below 0.05; R² = 0.24; predicted £53.19 (4 visits) and £74.58 (12 visits)

In [ ]:
# ➡️ E.2 Fit a second model with classes_per_month instead. Which single predictor explains more of the variation in spend?
classes_model = smf.ols('monthly_spend ~ classes_per_month', data=df).fit()

print(f"Coefficient: £{classes_model.params['classes_per_month']:.2f} per class")
print(f"p-value:     {classes_model.pvalues['classes_per_month']:.4f}")
print(f"R-squared:   {classes_model.rsquared:.2f}")

# ✅ £4.57 per class; R² = 0.25: slightly more than visits (0.24)

### 💬 What does it mean?

- Explain the visits coefficient in plain English.
- What does an R² of about 0.24 tell you?
- Why might one predictor not be enough? What else could explain spend?

**✏️ MODEL ANSWER:**

Each extra visit a month is linked to about **£2.67** more monthly spend (p below 0.05, so very unlikely to be chance): a member who visits 12 times a month is predicted to spend about **£75**, against **£53** for one who visits 4 times. Each extra class is linked to about **£4.57** more. But R² is only about **0.24** for visits (0.25 for classes): each explains only around a **quarter** of the variation in spend, so most of the difference between members is due to **other things**, such as personal training sessions and the membership fee. Visits and classes are also related to each other, so the two models overlap. To separate their effects, we need a model with **several predictors**: that is next week.

---
# Section F: Summarise
*about 5 minutes*

### ✏️ F.1 Key findings

- Write **three** findings for the manager: one about time, one about combinations of members, and one about spend.

**MODEL ANSWER:**

1. **Over time:** joins are steady (about 83 a month) and cancellations rose slowly as membership grew, to about 20 a month. Most cancellations happen in members' **first four months**. A spike in January 2025 (95 cancellations on 30 January) looks like a **data recording issue** and should be checked before it is reported.
2. **Combinations:** **Monthly Flex members who rarely visit** are most at risk (56.2% have cancelled), against 1.4% of Annual members who visit most.
3. **Spend:** each extra visit is linked to about **£2.67** more spend a month, but visits alone explain only about a quarter of the variation (R² = 0.24).

---
## 🔴 Extension 1: which single predictor explains the most?

Use a **loop** to fit a one-predictor model of `monthly_spend` for each of these columns, and print each R²: `visits_per_month`, `classes_per_month`, `pt_sessions_per_month`, `distance_km`, `age`. Which explains the most? Which explains nothing?

In [ ]:
# 🔴 Extension 1: compare predictors with a loop
for predictor in ['visits_per_month', 'classes_per_month', 'pt_sessions_per_month', 'distance_km', 'age']:
    m = smf.ols(f'monthly_spend ~ {predictor}', data=df).fit()
    print(f"{predictor:22} coefficient {m.params[predictor]:7.2f}   R² {m.rsquared:.3f}   p {m.pvalues[predictor]:.4f}")

# ✅ Personal training explains the most (R² 0.50, about £20.53 per session); age explains
#    nothing (R² 0.000, p 0.85). Note the f-string inside the formula

---
## 🔴 Extension 2: active members over time

How many members were **active** at the end of each month? A member is active at a date if they had joined by then and had **not yet cancelled**. Use a loop over the month-end dates.

In [ ]:
# 🔴 Extension 2: active members at the end of each month
month_ends = pd.date_range('2022-01-31', '2025-01-31', freq='ME')
active = []
for m in month_ends:
    joined = df['join_date'] <= m
    still_member = df['cancel_date'].isna() | (df['cancel_date'] > m)
    active.append((joined & still_member).sum())

plt.figure(figsize=(12, 5))
plt.plot(month_ends, active, marker='o')
plt.title('FitLife active members at the end of each month')
plt.xlabel('Month')
plt.ylabel('Active members')
plt.show()

# ✅ A steady upward trend: from 78 (January 2022) to 2,300 (January 2025)

---
## ✅ End of Task: Recap

You have taken your FitLife analysis further:

- **Created an outcome from a date**, and checked it: exactly the skill you need for NovaTel
- **Time series** of joins and cancellations, and **investigated a suspicious spike** before reporting it
- **Pivot tables** to see how two factors combine
- **Gemini as an assistant**: you generated code, explained it, and checked it
- Adding a **regression model**, and why one predictor is not enough

---

### 💾 Before you leave

Make sure your notebook is saved to your Google Drive, if you have not done so already:
- **File → Save a copy in Drive**
- Name it: `Week9_Task_FitLife_Time_Segments_Regression_YourName.ipynb`

---

### 🔮 Next week

Regression models with **several predictors**, and **logistic regression** for a yes/no outcome such as cancellation (or churn). Keep this notebook and your AI log.